In [1]:
# Setup
import math_toolkit
math_toolkit.activate()

# The Fourier premise - all functions are trig polynomials

The fundamental premise of Fourier analysis is:

:::{prf:theorem} Premise of Fourier analysis

Any periodic function $f:\mathbb{R}\to \mathbb{R}$ can be represented as a trigonometric series. 
For a 1-periodic function:

$$
f(x) \approx a_0 + \sum_{n=1}^{\infty}\left(a_n\cos(2\pi n x)+b_n\sin(2\pi n x)\right).
$$
:::


The game below uses only sine modes
$$
\sum_{n=1}^{5} b_n\sin(2\pi n x),
$$
and the challenge is to discover the coefficients by hand.

## The matching game

A hidden target function $f$ is generated as a sum of $N_{max}=5$ sine modes. Each target coefficient is chosen randomly from the grid

\begin{equation*}
-1,-0.9,-0.8,\ldots,0.8,0.9,1
\end{equation*}


Your model has the same five sine modes, but its coefficients are controlled by sliders with step size $0.01$. The figure reports the largest error:

\begin{equation}
\max_x \Bigl|f(x)-\sum_{n=1}^{N_{max}} b_n\sin(2 \pi n x) \Bigr|
\end{equation}


In [2]:
Nmax = 5

In [ ]:
import random

def create_mystery_sine_function(*, modes):
    """Return a randomly generated pure-sine target function."""
    mode_numbers = tuple(range(1, modes + 1))
    coefficient_choices = tuple(i / 10 for i in range(-10, 11))

    # Regenerate only the zero target, so the matching game always has a visible signal.
    target_coefficients = tuple(random.choice(coefficient_choices) for _ in mode_numbers)
    while all(coefficient == 0 for coefficient in target_coefficients):
        target_coefficients = tuple(random.choice(coefficient_choices) for _ in mode_numbers)

    target_expr = sum(
        Float(coefficient) * sin(2 * pi * mode * x)
        for mode, coefficient in zip(mode_numbers, target_coefficients)
    )
    return ImplementedFunction("f", target_expr >> Num(var=x))


In [3]:
f = create_mystery_sine_function(modes=Nmax)

r"""
A new hidden target has been generated. It has the form
$$
f(x)=\sum_{n=1}^{ {{ Nmax }} } b_n\sin(2\pi n x),
\qquad b_n\in\{-1,-0.9,-0.8,\ldots,0.9,1\}.
$$

The coefficients are deliberately not printed. The graph is the evidence. The name `f` is the numerical version compiled with `Num(var=x)` for the error calculation.
""">> md


A new hidden target has been generated. It has the form
```{math}
:enumerated: false
f(x)=\sum_{n=1}^{ 5 } b_n\sin(2\pi n x),
\qquad b_n\in\{-1,-0.9,-0.8,\ldots,0.9,1\}.
```

The coefficients are deliberately not printed. The graph is the evidence. The name `f` is the numerical version compiled with `Num(var=x)` for the error calculation.


In [4]:
# Create the student-controlled model with the same sine modes.
model_coefficients = tuple(b[i] for i in range(1, Nmax + 1))
g = Function("g")(x) @EqDef@ (Sum(b[n]*sin(2*pi*n*x) ,(n,1,Nmax)))

md(r"""
The adjustable model is
$$
g(x)={{ g(x)}}.
$$

The sliders below move the coefficients $b_1,\ldots,b_{{ Nmax }}$ in increments of $0.01$.
""")


The adjustable model is
```{math}
:enumerated: false
g(x)=\sum_{n=1}^{5} \sin{\left(2 \pi n x \right)} {b}_{n}.
```

The sliders below move the coefficients $b_1,\ldots,b_5$ in increments of $0.01$.


In [5]:
fig2 = figure("Match hidden sine series", new=True)
fig2.view.home_range = {"x": (-1/2, 1/2), "y": (-(Nmax + 0.5)/1.41, (Nmax + 0.5)/1.41)}
fig2.show()


In [6]:
fig2.plot(
        f(x),x,
        name="mystery",
        style={"width": 4, "opacity":0.6},
        samples=1001,
    )
fig2.plot(
        g(x),x,
        name="model",
        style={"width": 2, "opacity": 1},
        samples=1001,
    )
fig2.parameters({b[j]:{"value":0,"min":-1,"max":1,"step":0.01} for j in range(1,Nmax+1)})


In [7]:
fig2.info(
        "Largest error on $(-\\tfrac{1}{2}, \\tfrac{1}{2})$: ",
        LinftyNorm(f(x)-g(x),x,(-1/2,1/2)),
        name="sup-error",    )

Fine-tune the sliders to get the smallest possible error. 

:::{hint}
The Fourier coefficients of the mystery function have only one digit after the decimal point. 
:::

### Try the harder version

After matching the five-mode target, rerun the notebook with

```python
Nmax = 10
```

The premise is the same, but ten amplitudes are much harder to tune by eye. This is a useful classroom moment: Fourier coefficients turn the shape-matching problem into many smaller amplitude decisions, one frequency at a time.

## Recap

A pure sine wave has one amplitude and one frequency. Fourier analysis begins from the idea that periodic functions can be approximated by adding many such waves. In the game, the target and the model share the same frequencies, so matching the graph means finding the right amplitudes.

The reusable toolkit pattern is symbolic first, numeric when needed: write the sine series as an expression, use `plot(...)` for the live figure, express the matching score symbolically on the interval, and display that diagnostic with `fig.info(...)`.

## Related topics

- [Playing curve plots as sound](../tutorials/playing_curves.ipynb)
- [Square wave Fourier series](square_wave_fourier_series.ipynb)
- [figure](../library/figure.ipynb)
- [plot](../library/plot.ipynb)
- [Num](../library/Num.ipynb)